# Carousel Metric Analysis

This is the cleaned notebook-first version of the original `carousel_metric.ipynb`.

The goal is to keep the research workflow easy to read:

1. Prepare eye-tracking data before the first movie click.
2. Estimate empirical examination frequency for each carousel/movie position.
3. Generate candidate discount functions.
4. Compare candidates against empirical examination behavior.
5. Run the binary or graded N2DCG simulation.

The raw original notebook is preserved as `notebooks/carousel_metric_original.ipynb`.

## 0. Setup

Run this notebook from either the repository root or the `notebooks/` directory.

Expected local data files:

- `data/raw/summary_feedback.csv`
- `data/raw/click_summary_dataset.csv`

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INTERACTIONS_CSV = DATA_DIR / "summary_feedback.csv"
CLICKS_CSV = DATA_DIR / "click_summary_dataset.csv"

TARGET_GROUP = "uva"      # one of: "overall", "kinit", "uva"
N_TRIALS = 20_000
RNG_SEED = 42

print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {OUTPUT_DIR}")

In [ ]:
from carousel_metric.data import prepare_examination_results
from carousel_metric.discounts import candidate_discount_frames, candidate_display_names
from carousel_metric.metrics import score_candidate_discounts
from carousel_metric.plotting import (
    plot_candidate_comparison,
    plot_discount_heatmap,
    plot_examination_heatmap,
)
from carousel_metric.cli import examination_to_matrix
from carousel_metric.simulation import (
    SimulationConfig,
    format_simulation_report,
    run_simulation,
)

In [ ]:
missing = [path for path in [INTERACTIONS_CSV, CLICKS_CSV] if not path.exists()]

if missing:
    print("Missing data files:")
    for path in missing:
        print(f"- {path}")
    print("\nPlace the CSV files in data/raw/ before running the analysis cells.")
else:
    print("Data files found. Ready to run.")

## 1. Data Preparation

This reproduces the original notebook's cleaning logic:

- keep interactions before the first movie click;
- use only free-browsing tasks;
- keep only user-task pairs that visited the clicked movie before clicking;
- apply the original `Movie_Familiarity` filter;
- remove consecutive duplicate fixation positions;
- build normalized examination-frequency tables.

In [ ]:
examination_results = prepare_examination_results(
    interactions_csv=INTERACTIONS_CSV,
    clicks_csv=CLICKS_CSV,
    apply_familiarity_filter=True,
)

for group, frame in examination_results.items():
    output_path = OUTPUT_DIR / f"examination_{group}.csv"
    frame.to_csv(output_path, index=False)
    print(f"{group:>7}: {len(frame):>3} positions -> {output_path.name}")

examination_results[TARGET_GROUP].head()

## 2. Empirical Examination Heatmaps

The empirical heatmaps show normalized examination frequency by carousel row and movie position.

In [ ]:
for group, title in {
    "overall": "Overall Binary Examination",
    "kinit": "KINIT Binary Examination",
    "uva": "UvA Binary Examination",
}.items():
    frame = examination_results[group]
    if frame.empty:
        print(f"Skipping {group}: no rows after filtering.")
        continue

    plot_examination_heatmap(
        frame,
        title=title,
        output_path=OUTPUT_DIR / f"examination_{group}.pdf",
        show=True,
    )

## 3. Candidate Discount Functions

The six candidate discount functions from the original notebook are generated below.

In [ ]:
discounts = candidate_discount_frames()
display_names = candidate_display_names()

for key, frame in discounts.items():
    output_path = OUTPUT_DIR / f"discount_{key}.csv"
    frame.to_csv(output_path, index=False)
    print(f"{key:>32}: {output_path.name}")

list(discounts.keys())

In [ ]:
for key, frame in discounts.items():
    plot_discount_heatmap(
        frame,
        title=display_names[key],
        output_path=OUTPUT_DIR / f"discount_{key}.pdf",
        show=True,
    )

## 4. Candidate Scoring

Candidates are ranked by:

1. highest Spearman correlation;
2. highest Pearson correlation;
3. lowest MSE.

In [ ]:
metric_df = score_candidate_discounts(
    examination_results[TARGET_GROUP],
    discounts,
)

metric_df.to_csv(OUTPUT_DIR / "metrics_summary.csv", index=False)
metric_df

## 5. Final Comparison Figure

This recreates the original 2x3 comparison figure: empirical examination frequency versus each candidate discount function.

In [ ]:
_, metric_df = plot_candidate_comparison(
    examination_results[TARGET_GROUP],
    discounts,
    output_path=OUTPUT_DIR / "comparison_empirical_vs_candidate_discount_functions.pdf",
    show=True,
)

metric_df

## 6. Simulation: Original vs Reformulated N2DCG

The simulation compares how often the original and reformulated N2DCG agree with the empirical examination-based ground truth.

Use `relevance_mode="binary"` to reproduce the first simulation and `relevance_mode="graded"` for the graded version.

In [ ]:
p_exam = examination_to_matrix(examination_results[TARGET_GROUP])

binary_config = SimulationConfig(
    n_trials=N_TRIALS,
    rng_seed=RNG_SEED,
    relevance_mode="binary",
)

binary_result = run_simulation(p_exam, config=binary_config)
binary_report = format_simulation_report(binary_result)

(OUTPUT_DIR / "simulation_binary.txt").write_text(binary_report)
print(binary_report)

In [ ]:
graded_config = SimulationConfig(
    n_trials=N_TRIALS,
    rng_seed=RNG_SEED,
    relevance_mode="graded",
)

graded_result = run_simulation(p_exam, config=graded_config)
graded_report = format_simulation_report(graded_result)

(OUTPUT_DIR / "simulation_graded.txt").write_text(graded_report)
print(graded_report)

## 7. Outputs

Generated files are written to `outputs/`:

- examination CSVs and PDFs;
- candidate discount CSVs and PDFs;
- `metrics_summary.csv`;
- final comparison PDF;
- binary and graded simulation reports.

In [ ]:
for path in sorted(OUTPUT_DIR.glob("*")):
    if path.is_file() and path.name != ".gitkeep":
        print(path.relative_to(PROJECT_ROOT))